# Strojenie modelu

<div style="text-align: center;"><img src=".//Images//Hiperparametry.png" alt="Hiperparametry" width="400" height="120" style="margin: 10px; "/></div>

Proces strojenia obejmuje:
- estymator (np. klasyfikator lub regresor),
- przestrzeń parametrów,
- metodę przeszukiwania (pełna lub losowa),
- schemat walidacji krzyżowej,
- funkcję oceny (np. accuracy, f1, roc_auc).

# Dobór estymatora (modelu) — pierwszy krok strojenia

Wybór estymatora to kluczowy moment w projektowaniu modelu uczenia maszynowego. Od niego zależy, **jakie zależności w danych będą możliwe do uchwycenia**, jakie będą możliwości interpretacji wyników i ile zasobów obliczeniowych trzeba będzie wykorzystać.

Estymator to model, który **uczy się zależności** między cechami (`X`) a etykietą (`y`). W `scikit-learn` każdy estymator musi implementować metody `.fit()` (uczenie) i `.predict()` (predykcja).

---

**Jak wybieramy estymator?**

1. **Charakter danych**
   
- Czy dane są **liniowe czy nieliniowe**?  
  - Dane liniowe → prostsze modele jak `LogisticRegression`, `LinearRegression`.
  - Dane nieliniowe → modele nieliniowe jak `RandomForest`, `SVC`, `XGBoost`.
</br></br>
- Czy dane zawierają **zmienne kategorialne**?
  - Niektóre estymatory (np. drzewa decyzyjne, Random Forest) radzą sobie z nimi bez kodowania.
</br></br>
- Czy mamy **dużo cech** (tzn. duży wymiar)?
  - Modele liniowe dobrze działają w przestrzeniach wysokowymiarowych (np. `LogisticRegression` z regularizacją).
  - W przypadku bardzo wielu cech i mało danych, korzystne są modele z wbudowaną selekcją cech (np. Lasso).
</br></br>
2. **Rozmiar i jakość zbioru danych**
   
- **Małe zbiory danych**: unikamy złożonych modeli (mogą się przeuczyć).  
  Modele o mniejszej wariancji (np. regresja logistyczna, drzewa o małej głębokości) mogą być lepsze.

- **Duże zbiory danych**: można stosować bardziej złożone modele jak `RandomForest`, `GradientBoosting`.

- Jeśli dane są **niezbalansowane** warto rozważyć estymatory z obsługą wag klas (`class_weight='balanced'`) lub modele odporne na niezbalansowanie.

3. **Wymagana interpretowalność**
   
- Jeśli ważne jest zrozumienie działania modelu (np. w medycynie, finansach), lepiej wybrać:
  - modele liniowe,
  - drzewa decyzyjne (proste),
  - modele z możliwością obliczenia ważności cech.
</br></br>
- W zadaniach, gdzie **liczy się przede wszystkim dokładność predykcji**, a model nie musi być bezpośrednio interpretowalny, można sięgnąć po bardziej złożone estymatory (`RandomForest`, `XGBoost`, `SVC`).

4. **Ograniczenia obliczeniowe**

- SVC z jądrem RBF może być bardzo dokładny, ale **niewydajny przy dużych zbiorach danych**.
- Modele drzewiaste skalują się lepiej — `RandomForest` i `XGBoost` są efektywne i wspierają równoległość.

5. **Typ problemu**

- Jeśli celem jest **klasyfikacja binarna lub wieloklasowa** → dobieramy klasyfikator.
- Jeśli celem jest **regresja** (wartość liczbowa) → regresor.
- W przypadku zadań rankingowych, detekcji anomalii, grupowania — potrzebne są inne estymatory (`OneClassSVM`, `KMeans`, itp.).

---

**Przykład doboru estymatora**:

| Dane                               | Dobry wybór estymatora                        |
|------------------------------------|-----------------------------------------------|
| Mały zbiór, dane liniowe           | `LogisticRegression` z regularizacją          |
| Dane nieliniowe, z cechami kategorialnymi | `RandomForestClassifier`                |
| Duży zbiór, klasy niezbalansowane  | `XGBoostClassifier`, `RandomForest` z wagami |
| Dużo cech, mało przykładów         | `Lasso`, `LogisticRegression` z L1            |
| Potrzebna interpretowalność       | `DecisionTreeClassifier`, `LogisticRegression`|

## Przykład wyboru modelu (Bank Marketing)

<div style="text-align: center;"><img src=".//Images//Bank.png" alt="Bank" width="400" height="120" style="margin: 10px; "/></div>

Zbiór **Bank Marketing (bank-additional-full.csv)** pochodzi z rzeczywistej kampanii marketingowej portugalskiego banku. Celem jest przewidzenie, czy klient zgodzi się na **subskrypcję lokaty terminowej** (klasa `"yes"`), na podstawie danych o kliencie i historii kontaktów.

---

**Podstawowe informacje:**

| Cecha                         | Wartość                         |
|------------------------------|----------------------------------|
| Liczba przykładów            | 41188                            |
| Liczba cech (kolumn)         | 20 (bez targetu)                 |
| Typ zadania                  | Klasyfikacja binarna             |
| Cel (target)                 | `y`: `"yes"` (subskrybował) / `"no"` (nie) |
| Problem                      | **Niezbalansowany zbiór** (~11% "yes") |
| Źródło                       | UCI ML Repository                |


**Opis wybranych cech:**

**Dane demograficzne klienta**:
- `age` – wiek klienta,
- `job` – zawód (np. "admin.", "technician", "student"),
- `marital` – stan cywilny,
- `education` – poziom wykształcenia,
- `default` – czy klient ma niespłacony kredyt?,
- `housing` – czy ma kredyt hipoteczny?,
- `loan` – czy ma inny kredyt?

**Szczegóły ostatniej kampanii marketingowej**:
- `contact` – typ kontaktu (telefon komórkowy/stacjonarny),
- `month` – miesiąc kontaktu,
- `day_of_week` – dzień tygodnia kontaktu,
- `duration` – **czas trwania ostatniego kontaktu** (w sekundach)

**Historia kontaktów**:
- `campaign` – liczba kontaktów w tej kampanii,
- `pdays` – liczba dni od ostatniego kontaktu,
- `previous` – liczba wcześniejszych kontaktów,
- `poutcome` – wynik poprzedniej kampanii.

**Ekonomiczne wskaźniki (na poziomie makro)**:
- `emp.var.rate` – wskaźnik zatrudnienia,
- `cons.price.idx` – indeks cen konsumpcyjnych,
- `cons.conf.idx` – indeks zaufania konsumentów,
- `euribor3m` – stopa procentowa EURIBOR 3M,
- `nr.employed` – liczba zatrudnionych.

**Uwagi praktyczne:**
- **Zmienna `duration`** – uwaga: nie powinna być używana w modelach predykcyjnych, bo jest znana **po fakcie**.

**`duration`** to czas trwania ostatniego kontaktu z klientem (rozmowy telefonicznej). Jej wartość jest znana dopiero po kontakcie – a więc zależna od decyzji klienta.

    * jeśli klient powiedział "nie" → rozmowa była krótka,
    * jeśli "tak" → rozmowa była dłuższa,

Model może się tego łatwo nauczyć, osiągając nienaturalnie wysokie wyniki, ale nie będzie użyteczny w praktyce.

- **Zbiór niezbalansowany**: tylko ok. 11% klientów zgadza się na ofertę – dlatego warto stosować metryki takie jak `recall`, `roc_auc`, `f1`, oraz `class_weight='balanced'`.

Źródło: https://archive.ics.uci.edu/dataset/222/bank+marketing

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

In [2]:
# Wczytanie danych
df = pd.read_csv('./Data/bank-additional-full.csv', sep=';')

In [3]:
# Kodowanie zmiennych kategorialnych
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

In [4]:
# Usunięcie cechy 'duration'
X = df_encoded.drop(columns=['y', 'duration'])
y = df_encoded['y']

In [5]:
# Podział danych
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [6]:
# Lista estymatorów z uwzględnieniem przeskalowania tam, gdzie potrzebne
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    # 'SVC (linear kernel)': make_pipeline(StandardScaler(), SVC(kernel='linear', probability=True)), bardzo kosztowny obliczeniowa, jeśli dużo danych
    'Random Forest': RandomForestClassifier(),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier())
}

In [7]:
# Ocena każdego modelu
print("Porównanie estymatorów:")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else float('nan')

    print(f"\n{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Recall:   {rec:.4f}")
    print(f"  ROC AUC:  {roc:.4f}")

Porównanie estymatorów:

Logistic Regression
  Accuracy: 0.9013
  Recall:   0.2091
  ROC AUC:  0.7948

Random Forest
  Accuracy: 0.8943
  Recall:   0.2888
  ROC AUC:  0.7795

KNN
  Accuracy: 0.8947
  Recall:   0.2737
  ROC AUC:  0.7215


# Przestrzeń parametrów

Po wyborze estymatora kolejnym krokiem jest zdefiniowanie, które hiperparametry chcemy dostroić oraz jakie wartości dla nich sprawdzić. Ten zestaw nazywamy przestrzenią parametrów, a raczej hiperparametrów.

**Hiperparametry** to parametry, które nie są bezpośrednio uczone przez model. W `scikit-learn` przekazuje się je jako argumenty podczas tworzenia obiektu estymatora. Przykłady to: `C`, `kernel`, `gamma` w SVC, `alpha` w regresji Lasso itp.

Zaleca się przeszukiwanie przestrzeni hiperparametrów, aby znaleźć ustawienia dające najlepszy wynik walidacji krzyżowej.

Każdy parametr estymatora można w ten sposób optymalizować. Aby sprawdzić dostępne parametry i ich wartości domyślne, można użyć:

```python
estimator.get_params()
```

W `scikit-learn` dostępne są dwie główne metody:
- **`GridSearchCV`** – testuje wszystkie kombinacje zadanych wartości,
- **`RandomizedSearchCV`** – losuje ograniczoną liczbę zestawów parametrów.

Szybsze odpowiedniki tych metod to:
- **`HalvingGridSearchCV`**
- **`HalvingRandomSearchCV`**  

które stopniowo zawężają liczbę testowanych konfiguracji.

Niektóre estymatory mają własne, bardziej efektywne strategie strojenia, opisane w dokumentacji.

Istnieją również inne biblioteki, takie jak: 
* Optuna: https://optuna.org/
* Hyperopt: https://hyperopt.github.io/hyperopt/,
* Ray Tune: https://docs.ray.io/en/latest/tune/index.html,

które posiadają bardziej zaawansowaną kontrolą i wsparcie dla optymalizacji rozproszonej.

Warto pamiętać, że często tylko kilka hiperparametrów znacząco wpływa na jakość modelu – pozostałe mogą pozostać domyślne. Szczegółowe informacje znajdziesz w dokumentacji konkretnego estymatora.

## Najpopularniejsze metody optymalizacji hiperparametrów w `sklearn`:

### Grid Search (przeszukiwanie siatki)
Przegląd – testuje wszystkie kombinacje z zadanego zbioru parametrów.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=model, 
                           param_grid=param_grid, 
                           cv=5,                        # <----- kroswalidacja, można podać też schemat walidacji
                           scoring='accuracy') 
grid_search.fit(X_train, y_train)

print("Najlepsze parametry:", grid_search.best_params_)
print("Najlepszy wynik:", grid_search.best_score_)
```

---

### Randomized Search  
Zamiast testować wszystkie kombinacje, losuje losowy podzbiór – zwykle szybsza i mniej kosztowna obliczeniowo.

```python
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

model = RandomForestClassifier()
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

random_search = RandomizedSearchCV(estimator=model, 
                                   param_distributions=param_dist, 
                                   n_iter=20, 
                                   cv=5, 
                                   scoring='accuracy', 
                                   random_state=42)
random_search.fit(X_train, y_train)

print("Najlepsze parametry:", random_search.best_params_)
print("Najlepszy wynik:", random_search.best_score_)
```

---

### Bayesian Optimization 
Uczy się na podstawie poprzednich wyników – inteligentniejsze przeszukiwanie.

```python
from skopt import BayesSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
param_space = {
    'n_estimators': (50, 200),
    'max_depth': (5, 50),
    'min_samples_split': (2, 10)
}

opt = BayesSearchCV(estimator=model, 
                    search_spaces=param_space, 
                    n_iter=30, 
                    cv=5, 
                    scoring='accuracy', 
                    random_state=42)
opt.fit(X_train, y_train)

print("Najlepsze parametry:", opt.best_params_)
print("Najlepszy wynik:", opt.best_score_)
```

> Do użycia `BayesSearchCV` wymagany jest pakiet `scikit-optimize`: `pip install scikit-optimize`

---

**Dodatkowe wskazówki**:
- Używaj `Pipeline` jeśli optymalizujesz również preprocessing (np. skalowanie, PCA).
- Dobrą praktyką jest optymalizacja względem `roc_auc`, `f1`, `neg_log_loss` zamiast samej `accuracy`, szczególnie przy niezbalansowanych klasach.

## Przykład strojenia parametrów (Bank Marketing)

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [9]:
# Wczytanie danych
df = pd.read_csv('./Data/bank-additional-full.csv', sep=';')

In [10]:
# Kodowanie danych kategorialnych
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

In [11]:
# Podział danych
X = df_encoded.drop(columns=['y', 'duration'])
y = df_encoded['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [12]:
# Model domyślny
clf_default = RandomForestClassifier(random_state=42)
clf_default.fit(X_train, y_train)
y_pred_default = clf_default.predict(X_test)
y_prob_default = clf_default.predict_proba(X_test)[:, 1]

In [13]:
# Hiperparametry domyślne
print("Domyślne hiperparametry RandomForestClassifier:")
for param, value in clf_default.get_params().items():
    print(f"{param}: {value}")

Domyślne hiperparametry RandomForestClassifier:
bootstrap: True
ccp_alpha: 0.0
class_weight: None
criterion: gini
max_depth: None
max_features: sqrt
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 1
min_samples_split: 2
min_weight_fraction_leaf: 0.0
monotonic_cst: None
n_estimators: 100
n_jobs: None
oob_score: False
random_state: 42
verbose: 0
warm_start: False


In [14]:
# Wyniki domyślnego modelu
print("\n=== Wyniki domyślnego modelu ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_default):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_default):.4f}")
print(f"f1:       {f1_score(y_test, y_pred_default):.4f}")


=== Wyniki domyślnego modelu ===
Accuracy: 0.8963
Recall:   0.2888
f1:       0.3854


In [15]:
# GridSearchCV – ustawienia
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=5,
                           scoring='f1',
                           n_jobs=-1,  # <--- potrz Dodatki...
                           verbose=3)  # <--- pokazuje postęp strojenia, przy n_jobs=-1 może nie być zauważone, zmień na n_jobs=1

In [16]:
%%time
# GridSearchCV – strojenie hiperparametrów
grid_search.fit(X_train, y_train);

Fitting 5 folds for each of 24 candidates, totalling 120 fits
CPU times: total: 3.33 s
Wall time: 1min 12s


In [17]:
# Model po strojeniu
best_clf = grid_search.best_estimator_
y_pred_tuned = best_clf.predict(X_test)
y_prob_tuned = best_clf.predict_proba(X_test)[:, 1]

print("\n=== Wyniki po GridSearchCV ===")
print("Najlepsze parametry:", grid_search.best_params_)
print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_tuned):.4f}")
print(f"f1:       {f1_score(y_test, y_pred_tuned):.4f}")


=== Wyniki po GridSearchCV ===
Najlepsze parametry: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Accuracy: 0.8663
Recall:   0.6236
f1:       0.5124


## Strojenie hiperparametrów przy użyciu BayesSearchCV (scikit-optimize)

Zainstaluj pakiet scikit-optimize `pip install scikit-optimize (skopt)`.

Zamiast losowo testować parametry (jak w `RandomizedSearchCV`), `BayesSearchCV` uczy się, które kombinacje są bardziej obiecujące, i skupia się na nich.

Używa modelu probabilistycznego (np. Gaussian Process) do przewidywania wyników dla nowych kombinacji hiperparametrów.

W praktyce daje dobre wyniki przy mniejszej liczbie prób.

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

In [19]:
# Wczytanie danych
df = pd.read_csv('./Data/bank-additional-full.csv', sep=';')

In [20]:
# Kodowanie zmiennych kategorialnych
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

In [21]:
# Usunięcie cechy 'duration'
X = df_encoded.drop(columns=['y', 'duration'])
y = df_encoded['y']

In [22]:
# Podział danych
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [23]:
# Model bazowy (bez strojenia)
base_model = RandomForestClassifier(random_state=42)

In [24]:
base_model.fit(X_train, y_train)
y_pred_base = base_model.predict(X_test)
y_prob_base = base_model.predict_proba(X_test)[:, 1]

print("Przed strojeniem:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_base):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_prob_base):.4f}")
print(f"f1:       {f1_score(y_test, y_pred):.4f}")

Przed strojeniem:
Accuracy: 0.8963
Recall:   0.2888
ROC AUC:  0.7796
f1:       0.3694


In [25]:
# Przestrzeń hiperparametrów
param_space = {
    'n_estimators': Integer(100, 300),
    'max_depth': Integer(3, 15),
    'min_samples_split': Integer(2, 10),
    'class_weight': Categorical([None, 'balanced']),
}

In [26]:
# BayesSearchCV
opt = BayesSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    search_spaces=param_space,
    n_iter=25,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

In [27]:
%%time
# Dopasowanie modelu
opt.fit(X_train, y_train);

CPU times: total: 39.5 s
Wall time: 6min 19s


In [28]:
# Ocena najlepszego modelu
best_model = opt.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Najlepsze parametry:", opt.best_params_)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_prob):.4f}")
print(f"f1:       {f1_score(y_test, y_pred):.4f}")

Najlepsze parametry: OrderedDict({'class_weight': 'balanced', 'max_depth': 12, 'min_samples_split': 10, 'n_estimators': 112})
Accuracy: 0.8725
Recall:   0.6070
ROC AUC:  0.8120
f1:       0.5175


## Porównanie GridSearchCV i HalvingGridSearchCV

**HalvingGridSearchCV** i **HalvingRandomSearchCV** to przyspieszone metody strojenia hiperparametrów w scikit-learn, które stopniowo odrzucają słabsze konfiguracje, testując coraz mniej kandydatów na coraz większej liczbie danych.

**Używając parametru `aggressive_elimination`, można wymusić, aby proces przeszukiwania zakończył się z mniejszą liczbą kandydatów niż wartość `factor` w ostatniej iteracji.**

---

W `HalvingGridSearchCV` przeszukiwanie odbywa się etapami (iteracjami). W każdej iteracji:

- testowanych jest coraz **mniej kandydatów** (kombinacji hiperparametrów),
- każdy z nich trenowany jest na **coraz większej ilości danych**.

Domyślnie, przy `factor = 3`, w każdej iteracji odrzucane są 2/3 kandydatów, a 1/3 przechodzi dalej. Oznacza to, że **ostatnia iteracja** (czyli najdokładniejsze testowanie) **zawiera dokładnie `factor` kandydatów**.

Ale jeśli ustawi się:

```python
HalvingGridSearchCV(..., factor=3, aggressive_elimination=True)
```

to mechanizm będzie **bardziej "agresywny"** – **może odrzucać więcej kandydatów w każdej rundzie**, przez co do ostatniej iteracji przejdzie **mniej niż `factor` kandydatów**.

- **Przyspiesza** przeszukiwanie (mniej kosztownych testów w końcowych iteracjach),
- Można go użyć, jeśli mamy **dużą liczbę kombinacji parametrów**, ale chcemy skrócić czas przeszukiwania,
- Kosztem może być **nieco mniejsza dokładność** (bo mniej kandydatów testowanych będzie "do końca").

In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder
from time import time

In [30]:
from sklearn.experimental import enable_halving_search_cv  # <-- funkcja eksperymentalna, trzeba jawnie się na to zgodzić
from sklearn.model_selection import HalvingGridSearchCV

In [31]:
# Wczytanie i przygotowanie danych
df = pd.read_csv('./Data/bank-additional-full.csv', sep=';')

In [32]:
# Kodowanie zmiennych kategorialnych
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

In [33]:
# Usunięcie cechy duration
X = df_encoded.drop(columns=['y', 'duration'])
y = df_encoded['y']

# Podział danych
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [34]:
# Przestrzeń parametrów
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5],
    'class_weight': [None, 'balanced']
}

In [35]:
# GridSearchCV
print("== GridSearchCV ==")
start_g = time()
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train, y_train)
end_g = time()

best_g = grid.best_estimator_
y_pred_g = best_g.predict(X_test)
y_prob_g = best_g.predict_proba(X_test)[:, 1] # <--- dla roc

print("Najlepsze parametry:", grid.best_params_)
print(f"f1: {f1_score(y_test, y_pred_g):.4f}")
print(f"Czas wykonania: {end_g - start_g:.2f} s\n")

== GridSearchCV ==
Najlepsze parametry: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
f1: 0.5100
Czas wykonania: 47.25 s



In [36]:
# HalvingGridSearchCV
print("== HalvingGridSearchCV ==")
start_h = time()
halving = HalvingGridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    factor=2,  # domyślnie 3 – zmniejszamy na potrzeby porównania
    random_state=42,
    aggressive_elimination=False
)
halving.fit(X_train, y_train)
end_h = time()

best_h = halving.best_estimator_
y_pred_h = best_h.predict(X_test)
y_prob_h = best_h.predict_proba(X_test)[:, 1] # <--- dla roc

print("Najlepsze parametry:", halving.best_params_)
print(f"f1: {f1_score(y_test, y_pred_h):.4f}")
print(f"Czas wykonania: {end_h - start_h:.2f} s")

== HalvingGridSearchCV ==
Najlepsze parametry: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200}
f1: 0.4694
Czas wykonania: 19.89 s


# Podsumowanie

Co jeśli nie wiemy, które modele, metody i hiperparametry dobrać?
* Zajrzyj do dokumentacji estymatora (help(model), model.get_params()).
* Sprawdź domyślne wartości i przetestuj ich wpływ na małym zbiorze.
* Skorzystaj z literatury lub przykładów praktycznych (np. benchmarków na Kaggle).

Warto zerknąć do: https://scikit-learn.org/stable/modules/grid_search.html

# Dodatek 1 - time

W Jupyter Notebook można w bardzo prosty sposób sprawdzić czas wykonania fragmentu kodu przy użyciu **magicznych komend IPython**:

---

⏱️ **`%time`** – czas wykonania pojedynczej instrukcji

```python
%time model.fit(X_train, y_train)
```

Przydatne, gdy chcesz zmierzyć **pojedynczą operację**, np. `fit()` lub `predict()`.

---

⏱️ **`%timeit`** – uśredniony czas wykonania (kilka powtórzeń)

```python
%timeit model.predict(X_test)
```

Działa najlepiej do bardzo szybkich operacji, bo powtarza je wiele razy i podaje **średni czas i odchylenie standardowe**.

---

⏱️ **`%%time`** – czas wykonania całej komórki

Umieszczasz na **początku komórki**, jeśli chcesz zmierzyć czas wykonania **całego bloku kodu**:

```python
%%time

opt.fit(X_train, y_train)
y_pred = opt.predict(X_test)
```

---

⏱️ Pomiar czasu ręcznie

```python
import time

start = time.time()
opt.fit(X_train, y_train)
end = time.time()

print(f"Czas wykonania: {end - start:.2f} sekund")
```

---

**`CPU times: total: CZAS`**

- To **łączny czas pracy procesora** (suma czasu spędzonego przez wszystkie wątki/procesy na wykonywaniu kodu).
- Jeśli masz równoległość (np. `n_jobs=-1`), czas CPU może być większy niż rzeczywisty czas oczekiwania.

**Przykład:**
- Masz 4 wątki, każdy pracuje 30 sekund → `CPU time = 2 min` (4 × 30 s)

---

**`Wall time: CZAS`**

- To **rzeczywisty czas upłynięty na zegarze ściennym**, czyli czas, który **Ty czekano**, aż operacja się zakończy.
- Uwzględnia czasy oczekiwania, synchronizację, opóźnienia, itp.

---

- **`Wall time`** – do oceny, ile realnie trwa zadanie (np. strojenie modelu),
- **`CPU time`** – do analizy, ile procesorów zużyto (ważne przy `n_jobs > 1` lub ocenie kosztu obliczeniowego).

---

`tqdm` to **bardzo popularna biblioteka Pythona**, która umożliwia **szybkie dodanie paska postępu** do dowolnej pętli — bez konieczności pisania własnego kodu śledzącego postęp.

- Pokazuje **pasek postępu** dla dowolnej iteracji: `for`, `map()`, `list comprehension`, `DataFrame.apply()` itd.
- Informuje o:
  - liczbie wykonanych kroków,
  - procentowym postępie,
  - czasie działania i szacowanym czasie do końca (ETA).

---

**Instalacja**: ```pip install tqdm ```

**Przykład użycia**:

```python
from tqdm import tqdm
import time

for i in tqdm(range(100)):
    time.sleep(0.05)
```

W Jupyter Notebooku:

```python
from tqdm.notebook import tqdm
```

---

**Dokumentacja**:

- Oficjalna strona: [https://tqdm.github.io](https://tqdm.github.io)
- Repozytorium GitHub: [https://github.com/tqdm/tqdm](https://github.com/tqdm/tqdm)

# Dodatek 2 - przetwarzanie równoległe

Niektóre estymatory i narzędzia `scikit-learn` wykorzystują **równoległe przetwarzanie**, aby przyspieszyć obliczenia na wielu rdzeniach CPU.

Równoległość może być realizowana na trzy sposoby:

1. **joblib** – sterowana przez parametr `n_jobs`, używana przez większość estymatorów (wysokopoziomowa),
2. **OpenMP** – niskopoziomowa równoległość w kodzie C/Cython (nie sterowana `n_jobs`),
3. **BLAS / LAPACK** – równoległość w operacjach macierzowych NumPy/SciPy (także niezależna od `n_jobs`).

Niektóre estymatory mogą wykorzystywać wszystkie trzy formy równoległości w różnych etapach działania. Parametr `n_jobs` wpływa tylko na joblib, natomiast inne formy kontrolowane są przez zmienne środowiskowe lub `threadpoolctl`.

Szczegóły działania każdej metody opisane są sekcjach dokumentacji: https://scikit-learn.org/stable/computing/parallelism.html

---

Parametr `n_jobs` jest bardzo przydatny w **strojeniach hiperparametrów**, ponieważ pozwala przyspieszyć obliczenia przez równoległe uruchamianie eksperymentów (np. testów różnych zestawów parametrów w `GridSearchCV`).

- `n_jobs` = liczba równoległych procesów.
- `n_jobs=-1` = użyj **wszystkich dostępnych rdzeni CPU**.
- Stosowany w:
  - `GridSearchCV`
  - `RandomizedSearchCV`
  - `HalvingGridSearchCV`
  - oraz bezpośrednio w niektórych modelach (np. `RandomForestClassifier`).

Domyślna wartość parametru `n_jobs` w `scikit-learn` to:

**`n_jobs=None`**

- W większości klas (`GridSearchCV`, `RandomForestClassifier`, itp.) **`n_jobs=None` oznacza użycie tylko jednego rdzenia CPU**, czyli **brak równoległości**.